# SmartFlow — 01 · Environment setup & dataset preparation

**Scope of this notebook:** environment checks, dataset acquisition, annotation inspection, conversion to the YOLO detection format, split creation, and visual verification. **It does not train any model.** The last cell writes a *data gate* report; the training notebook must refuse to run unless that gate passed.

**Datasets handled** (from the dataset audit, `docs/dataset_audit.md`):

| Dataset | Status here | Why |
|---|---|---|
| UA-DETRAC — Hugging Face mirror `blues111/ua_detrac` | **Prepared automatically** (1 class: `vehicle`) | Public, ungated; real boxes; file checksums published by the Hub. The mirror only keeps `vehicle` + `ignored`. |
| BDD100K — Detection 2020 labels | **Prepared from files you provide** (4 classes) | Real multi-class boxes. Official archive filenames are not documented, so you download them from the official site and the notebook identifies them by the **md5 published in the official docs**. |
| UA-DETRAC — official XML annotations | **Optional, off by default** (4 classes) | Has `vehicle_type`, but is distributed via the authors' Google Drive with no published checksums. The XML schema is validated at runtime before any conversion. |
| QMUL Junction | **Excluded** | Single video, no bounding boxes (only clip-level event labels). |
| AI City Challenge 2021 | **Excluded** | Request-form access; T1 counts / T2 ReID crops / T4 anomaly timestamps are not detection boxes; T3 boxes are class-agnostic and incomplete. |

**How to run:** edit only **cell 1** if needed, then *Runtime → Run all*. Every cell is safe to rerun: downloads resume, verified archives are not re-hashed, extractions are skipped when already done, and prepared datasets are restored from a Drive cache when the configuration has not changed. If a cell stops with **`UserActionRequired`**, read the message: it says exactly what to upload or set.

## 1 · Configuration
The only cell you should need to edit. Every value that affects the prepared data is part of a fingerprint, so changing it automatically invalidates stale caches instead of silently reusing them.

Verified constants (do not edit unless the source changes):
* UA-DETRAC mirror: repository revision, file sizes and sha256 values come from the Hugging Face API for revision `206bb8f…`.
* BDD100K: md5 values are copied from the official `download.rst` in `github.com/bdd100k/bdd100k`.

In [ ]:
# 1. Configuration
from pathlib import Path

CFG = {
    "project": "SmartFlow",
    # Persistent storage (Google Drive). Raw archives, reports, plots, logs and the
    # prepared-dataset cache live here and survive Colab restarts.
    "drive_mount": "/content/drive",
    "drive_root": "/content/drive/MyDrive/SmartFlow",
    # Fast local disk for extracted images and YOLO datasets (wiped when the runtime ends).
    "local_root": "/content/smartflow_work",
    "seed": 42,
    "install_dependencies": True,
    "ultralytics_version": "8.4.160",   # pinned; released 2026-09-22
    "persist_prepared_to_drive": True,   # tar each validated YOLO dataset to Drive for fast restore
    "force_rebuild": False,              # True = ignore caches and rebuild from raw archives
    "sample_decode_per_split": 200,      # images fully decoded per split as a corruption spot-check
    "content_hash_leakage_check": True,  # md5 every prepared image to detect duplicates across splits
    "enable": {
        "ua_detrac_mirror": True,
        "bdd100k": True,
        "ua_detrac_official": False,
    },

    # --- UA-DETRAC, Hugging Face mirror (verified in the audit) -------------------
    "ua_detrac_mirror": {
        "repo": "blues111/ua_detrac",
        "revision": "206bb8fa86cdbb6c8676c4d81ddf2b5dd43b5654",
        "files": {
            "ua_detrac_training_set.zip": {
                "size": 5630247258,
                "sha256": "7045ffc28bb247248e2c16f8329c7131751fe1135bc96e327aef8816508ffaf9",
                "official_split": "train",
            },
            "ua_detrac_test_set.zip": {
                "size": 4252838102,
                "sha256": "ab018f74b2ed84bee47593871771887b771c7d79b6fb4fea0be5a89bfba77edd",
                "official_split": "test",
            },
        },
        # Counted from the archives' central directories during the audit.
        "expected": {"train": {"sequences": 60, "images": 83791},
                     "test": {"sequences": 40, "images": 56340}},
        "classes": ["vehicle"],
        # Every annotation class in the CSVs must have an explicit policy.
        "class_policy": {"vehicle": "map:vehicle", "ignored": "ignore_region"},
        "val_fraction": 0.15,   # share of official-train SEQUENCES held out for validation
        "frame_stride": 10,     # keep every Nth frame (25 FPS video => near-duplicate frames)
        "include_frames_without_rows": False,  # frames with no CSV row: unknown if empty -> excluded & reported
        "mask_value": 114,      # grey fill for ignored regions (YOLO has no 'ignore' label)
    },

    # --- BDD100K ------------------------------------------------------------------
    "bdd100k": {
        "auto_download": True,
        # Automatic source: the BDD100K download server at UC Berkeley (dl.yf.io, run by the dataset's
        # lead author). The ETH server named in the docs (dl.cv.ethz.ch) has been reported unreachable
        # since March 2026. Checked on 2026-10-21: directory listing, file sizes, archive contents,
        # and the labels md5 (measured by downloading the file). Tried in order; the IP is the same
        # host and is used when the dl.yf.io name does not resolve.
        "auto_source": {
            "base_urls": ["http://dl.yf.io/bdd100k/", "http://128.32.162.150/bdd100k/"],
            "files": {
                "images_100k": {"file": "bdd100k_images_100k.zip", "size": 5669071832, "md5": None},
                "labels_2018": {"file": "bdd100k_labels.zip", "size": 189638612,
                                "md5": "2d0165fd89b11025206e21a25ae1089d"},
            },
        },
        # Official package md5s from download.rst: lets you upload official files yourself instead.
        # 'Detection 2020 Labels' is used in preference to the 2018 labels whenever it is present.
        "official_archives": {
            "images_100k": {"md5": "5a0359c86a0b8713adab1eee9a3041cb", "doc_size": "5.3GB",
                            "doc_name": "100K Images"},
            "det_20_labels": {"md5": "b86a3e1b7edbcad421b7dad2b3987c94", "doc_size": "53MB",
                              "doc_name": "Detection 2020 Labels"},
        },
        "classes": ["car", "bus", "truck", "motorcycle"],
        # map:<class> -> becomes that class | drop -> not a SmartFlow target, removed and counted
        # mask -> vehicle-like but excluded; region greyed out so it is not an unlabelled vehicle
        # Any category found in the labels that is NOT listed here stops the notebook.
        "category_policy": {
            # names used by the 2020 labels
            "car": "map:car", "bus": "map:bus", "truck": "map:truck",
            "motorcycle": "map:motorcycle", "trailer": "map:truck",
            "pedestrian": "drop", "rider": "drop", "bicycle": "drop",
            "traffic light": "drop", "traffic sign": "drop", "other person": "drop",
            "train": "mask", "other vehicle": "mask",
            # names used by the 2018 labels (same objects, older names)
            "motor": "map:motorcycle", "person": "drop", "bike": "drop",
        },
        "val_fraction": 0.10,   # of official-train images (one image per video => video-level split)
        "allow_missing_images": False,
        "expected_images": {"train": 70000, "val": 10000, "test": 20000},  # verified in the archive listing
        "mask_value": 114,
    },

    # --- UA-DETRAC, official annotations (optional, provided by you) --------------
    "ua_detrac_official": {
        "classes": ["car", "bus", "truck"],
        "vehicle_type_policy": {"car": "map:car", "van": "map:car", "bus": "map:bus",
                                "others": "map:truck"},
        "frame_stride": 10,
        "mask_value": 114,
    },
}
print("Configuration loaded. Enabled datasets:",
      [k for k, v in CFG["enable"].items() if v])

## 2 · Hardware and software report
Prints GPU, CPU, RAM, disk and the preinstalled framework versions. A missing GPU is a warning only, since nothing in this notebook needs one, but training will.

In [ ]:
# 2. Hardware / software report
import os, platform, shutil, subprocess, sys
from pathlib import Path
assert "CFG" in globals(), "Run cell 1 first."

def _run(cmd):
    try:
        return subprocess.run(cmd, capture_output=True, text=True, timeout=30).stdout.strip()
    except Exception as e:
        return f"unavailable ({e.__class__.__name__})"

HW = {"python": sys.version.split()[0], "platform": platform.platform(),
      "cpu_count": os.cpu_count()}
try:
    with open("/proc/meminfo") as f:
        HW["ram_gb"] = round(int(f.readline().split()[1]) / 1024**2, 1)
except OSError:
    HW["ram_gb"] = None
_disk_path = "/content" if Path("/content").exists() else "."
_du = shutil.disk_usage(_disk_path)
HW["local_disk_free_gb"] = round(_du.free / 1024**3, 1)

HW["nvidia_smi"] = _run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                         "--format=csv,noheader"]) if shutil.which("nvidia-smi") else "nvidia-smi not found"
try:
    import torch
    HW["torch"] = torch.__version__
    HW["cuda_available"] = torch.cuda.is_available()
    HW["torch_cuda"] = torch.version.cuda
    HW["cudnn"] = torch.backends.cudnn.version() if torch.cuda.is_available() else None
    HW["gpu"] = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
except ImportError:
    HW["torch"], HW["cuda_available"] = None, False

for k, v in HW.items():
    print(f"{k:>20}: {v}")
if not HW["cuda_available"]:
    print("\nWARNING: no CUDA GPU visible. Data preparation works on CPU, but before training "
          "select Runtime -> Change runtime type -> GPU.")
if HW["local_disk_free_gb"] < 40:
    print(f"\nWARNING: only {HW['local_disk_free_gb']} GB free locally. UA-DETRAC (~10 GB extracted) "
          "plus BDD100K (~6 GB extracted) plus YOLO copies need roughly 25-30 GB.")

## 3 · Pinned dependencies
Only `ultralytics` is installed, pinned to `CFG["ultralytics_version"]`. Its requirements are loose (e.g. `torch>=1.8`), so pip keeps Colab's preinstalled torch, numpy and opencv. The cell compares package versions before and after the install. If anything preinstalled changed, it tells you to restart the runtime, and runs `pip check`. Everything else used here (numpy, pandas, Pillow, matplotlib, PyYAML, requests) ships with Colab.

In [ ]:
# 3. Install pinned dependencies (skipped when already satisfied)
import importlib.metadata as md, subprocess, sys
assert "CFG" in globals(), "Run cell 1 first."

WATCH = ["torch", "torchvision", "numpy", "opencv-python", "opencv-python-headless", "pillow",
         "matplotlib", "pandas", "pyyaml", "requests", "ultralytics"]

def _ver(pkg):
    try:
        return md.version(pkg)
    except md.PackageNotFoundError:
        return None

before = {p: _ver(p) for p in WATCH}
target = CFG["ultralytics_version"]
if not CFG["install_dependencies"]:
    print("CFG['install_dependencies'] is False -> skipping install.")
elif before["ultralytics"] == target:
    print(f"ultralytics=={target} already installed -> nothing to do.")
else:
    print(f"Installing ultralytics=={target} (currently: {before['ultralytics']}) ...")
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"ultralytics=={target}"],
                       capture_output=True, text=True)
    print(r.stdout[-2000:], r.stderr[-2000:])
    assert r.returncode == 0, "pip install failed - see the output above."

after = {p: _ver(p) for p in WATCH}
changed = {p: (before[p], after[p]) for p in WATCH if p != "ultralytics" and before[p] != after[p]}
for p in WATCH:
    print(f"{p:>24}: {after[p]}")
if CFG["install_dependencies"]:
    assert after["ultralytics"] == target, f"ultralytics is {after['ultralytics']}, expected {target}"
if changed:
    print("\nWARNING: the install changed preinstalled packages:", changed,
          "\n-> Runtime -> Restart session, then run from cell 1 again (the install will be skipped).")
chk = subprocess.run([sys.executable, "-m", "pip", "check"], capture_output=True, text=True)
print("\npip check:", chk.stdout.strip() or "OK", chk.stderr.strip())
DEP_VERSIONS = after

## 4 · Mount Google Drive
This uses the standard `google.colab.drive.mount` flow, and Colab asks you to authorise access. Outside Colab, which is only useful for testing, the notebook falls back to a local folder and says so.

In [ ]:
# 4. Mount Google Drive
from pathlib import Path
assert "CFG" in globals(), "Run cell 1 first."
try:
    from google.colab import drive as _gdrive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if Path(CFG["drive_mount"], "MyDrive").is_dir():
        print("Drive already mounted at", CFG["drive_mount"])
    else:
        _gdrive.mount(CFG["drive_mount"])
    assert Path(CFG["drive_mount"], "MyDrive").is_dir(), "Drive mount failed - rerun this cell."
else:
    if str(CFG["drive_root"]).startswith("/content/drive"):
        CFG["drive_root"] = str(Path.cwd() / "smartflow_drive_fallback")
        CFG["local_root"] = str(Path.cwd() / "smartflow_work")
    print("Not running in Colab: using local folders instead of Google Drive:\n ", CFG["drive_root"])

## 5 · Project directories and environment snapshot
Persistent folders on Drive: `raw/`, `prepared_cache/`, `checkpoints/`, `logs/`, `metrics/`, `plots/`, `reports/`. Fast local folders: `extracted/` and `yolo/`. The hardware and package versions plus a full `pip freeze` are saved to `logs/` for reproducibility.

In [ ]:
# 5. Directories + environment snapshot
import json, subprocess, sys
from datetime import datetime, timezone
from pathlib import Path
assert "CFG" in globals() and "HW" in globals(), "Run cells 1-4 first."

DRIVE_ROOT, LOCAL_ROOT = Path(CFG["drive_root"]), Path(CFG["local_root"])
DIRS = {
    "raw": DRIVE_ROOT / "raw",
    "prepared_cache": DRIVE_ROOT / "prepared_cache",
    "checkpoints": DRIVE_ROOT / "checkpoints",
    "logs": DRIVE_ROOT / "logs",
    "metrics": DRIVE_ROOT / "metrics",
    "plots": DRIVE_ROOT / "plots",
    "reports": DRIVE_ROOT / "reports",
    "local_extracted": LOCAL_ROOT / "extracted",
    "local_yolo": LOCAL_ROOT / "yolo",
}
for p in DIRS.values():
    p.mkdir(parents=True, exist_ok=True)
for k, p in DIRS.items():
    assert p.is_dir(), f"Could not create {p}"
    print(f"{k:>16}: {p}")

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
snapshot = {"utc": stamp, "hardware": HW, "packages": globals().get("DEP_VERSIONS"), "config": CFG}
(DIRS["logs"] / f"environment_{stamp}.json").write_text(json.dumps(snapshot, indent=2, default=str))
freeze = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True).stdout
(DIRS["logs"] / f"pip_freeze_{stamp}.txt").write_text(freeze)
print(f"\nEnvironment snapshot written to {DIRS['logs']} ({len(freeze.splitlines())} packages frozen).")

## 6 · Shared helpers
Everything later cells rely on lives here:
* **Hashing** with a sidecar cache, so a 5 GB archive is hashed once.
* **Resumable HTTP download** using `Range` requests and a `.part` file, with retries and clear errors on 401/403/404. It never tries to get around access control.
* **Safe zip extraction**: path-traversal checks, CRC verification, a disk-space check, an atomic temp-dir rename, and an "already extracted" marker.
* **`Report`**, which records PASS/WARN/FAIL checks to `reports/*.json` and raises on fatal failures.
* **YOLO writing, validation and sample-grid visualisation.**
* **Prepared-dataset cache** on Drive, keyed by a configuration fingerprint.

In [ ]:
# 6. Shared helpers
import hashlib, json, math, os, random, re, shutil, tarfile, time, zipfile
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
from PIL import Image

assert "DIRS" in globals(), "Run cell 5 first."
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp"}
CONVERTER_VERSION = "1"       # bump when conversion logic changes -> invalidates caches
STATE = globals().get("STATE") or {}


class UserActionRequired(RuntimeError):
    """The notebook cannot continue without something only you can provide."""


def log(msg):
    line = f"[{datetime.now().strftime('%H:%M:%S')}] {msg}"
    print(line)
    with open(DIRS["logs"] / "data_prep.log", "a") as f:
        f.write(line + "\n")


def human(n):
    for unit in ("B", "KB", "MB", "GB", "TB"):
        if abs(n) < 1024 or unit == "TB":
            return f"{n:.1f} {unit}"
        n /= 1024


def stable_hash(obj):
    return hashlib.sha256(json.dumps(obj, sort_keys=True, default=str).encode()).hexdigest()[:16]


class Report:
    """Collects named checks; fatal failures raise in finish()."""

    def __init__(self, name):
        self.name, self.checks, self.info = name, [], {}

    def check(self, label, ok, detail="", fatal=True):
        ok = bool(ok)
        self.checks.append({"check": label, "ok": ok, "fatal": fatal, "detail": str(detail)[:2000]})
        mark = "PASS" if ok else ("FAIL" if fatal else "WARN")
        print(f"  [{mark}] {label}" + (f" -> {detail}" if detail not in ("", None) else ""))
        return ok

    def failures(self):
        return [c for c in self.checks if c["fatal"] and not c["ok"]]

    def save(self):
        path = DIRS["reports"] / f"{self.name}.json"
        path.write_text(json.dumps({"name": self.name,
                                    "utc": datetime.now(timezone.utc).isoformat(),
                                    "passed": not self.failures(), "info": self.info,
                                    "checks": self.checks}, indent=2, default=str))
        return path

    def finish(self):
        path, fails = self.save(), self.failures()
        assert not fails, (f"{len(fails)} fatal check(s) failed in '{self.name}': "
                           + "; ".join(c["check"] for c in fails) + f". Full report: {path}")
        print(f"  => all fatal checks passed ({len(self.checks)} checks). Report: {path}")


def file_hash(path, algo="sha256", chunk=16 << 20):
    """Hash a file; cache the result next to it, keyed by size + mtime."""
    path = Path(path)
    st = path.stat()
    side = path.with_name(path.name + f".{algo}.json")
    if side.exists():
        try:
            c = json.loads(side.read_text())
            if c["size"] == st.st_size and c["mtime_ns"] == st.st_mtime_ns:
                return c["hash"]
        except (ValueError, KeyError):
            pass
    h, done, nxt = hashlib.new(algo), 0, 1 << 30
    log(f"Computing {algo} of {path.name} ({human(st.st_size)}) ...")
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
            done += len(block)
            if done >= nxt:
                print(f"    {human(done)} / {human(st.st_size)}")
                nxt += 1 << 30
    digest = h.hexdigest()
    side.write_text(json.dumps({"size": st.st_size, "mtime_ns": st.st_mtime_ns, "hash": digest}))
    return digest


def verify_file(path, size=None, md5=None, sha256=None):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Expected file is missing: {path}")
    actual = path.stat().st_size
    if size is not None and actual != size:
        raise IOError(f"Size mismatch for {path.name}: {actual} bytes on disk, {size} expected. "
                      "The file is incomplete or corrupt. Rerun the download (it resumes) or delete it.")
    for algo, expected in (("md5", md5), ("sha256", sha256)):
        if expected:
            got = file_hash(path, algo)
            if got.lower() != expected.lower():
                raise IOError(f"{algo} mismatch for {path.name}: got {got}, expected {expected}. "
                              f"The file is corrupt or not the expected file. Delete {path} and download again.")
    log(f"Verified {path.name}: size={human(actual)}"
        + (f", md5 ok" if md5 else "") + (f", sha256 ok" if sha256 else ""))
    return True


def download_resumable(url, dest, expected_size=None, retries=6, chunk=8 << 20, timeout=60):
    """HTTP download into dest.part with Range-resume; renames to dest when complete."""
    import requests
    dest = Path(dest)
    part = dest.with_name(dest.name + ".part")
    if dest.exists() and (expected_size is None or dest.stat().st_size == expected_size):
        log(f"Already downloaded: {dest.name} ({human(dest.stat().st_size)})")
        return dest
    if dest.exists():
        log(f"{dest.name} has the wrong size; treating it as a partial download.")
        dest.replace(part)
    dest.parent.mkdir(parents=True, exist_ok=True)
    for attempt in range(1, retries + 1):
        have = part.stat().st_size if part.exists() else 0
        if expected_size is not None and have > expected_size:
            log("Partial file is larger than expected; discarding it.")
            part.unlink()
            have = 0
        if expected_size is not None and have == expected_size:
            break
        headers = {"Range": f"bytes={have}-"} if have else {}
        try:
            with requests.get(url, headers=headers, stream=True, timeout=timeout,
                              allow_redirects=True) as r:
                if r.status_code in (401, 403):
                    raise UserActionRequired(
                        f"HTTP {r.status_code} for {url}: the server requires authorisation. "
                        "Download the file yourself through the official site and upload it "
                        f"to {dest.parent}. This notebook will not bypass access controls.")
                if r.status_code == 404:
                    raise UserActionRequired(f"HTTP 404 for {url}: the URL is wrong or the file moved. "
                                             "Check the official download page.")
                if r.status_code == 416 and have:
                    break  # requested range beyond end: the partial file is already complete
                r.raise_for_status()
                if have and r.status_code != 206:
                    log("Server ignored the Range header; restarting from byte 0.")
                    have = 0
                total = expected_size or (have + int(r.headers.get("Content-Length", 0))) or None
                log(f"Downloading {dest.name}: starting at {human(have)}"
                    + (f" of {human(total)}" if total else ""))
                nxt = have + (512 << 20)
                with open(part, "ab" if have else "wb") as f:
                    for block in r.iter_content(chunk):
                        f.write(block)
                        have += len(block)
                        if have >= nxt:
                            print(f"    {human(have)}" + (f" / {human(total)}" if total else ""))
                            nxt += 512 << 20
            if expected_size is None or have == expected_size:
                break
            log(f"Connection ended early at {human(have)}; resuming.")
        except UserActionRequired:
            raise
        except (requests.RequestException, OSError) as e:
            wait = min(2 ** attempt, 60)
            log(f"Attempt {attempt}/{retries} failed: {e}. Retrying in {wait}s (download resumes).")
            time.sleep(wait)
    else:
        raise IOError(f"Download did not complete after {retries} attempts: {url}. "
                      f"Partial data is kept in {part}; rerun the cell to resume.")
    got = part.stat().st_size
    if expected_size is not None and got != expected_size:
        raise IOError(f"Downloaded {got} bytes, expected {expected_size}. Rerun the cell to resume.")
    part.replace(dest)
    log(f"Download complete: {dest.name} ({human(got)})")
    return dest


def extract_archive_once(archive, dest, fingerprint):
    """Extract a zip/tar into dest exactly once (marker + fingerprint), safely and atomically."""
    archive, dest = Path(archive), Path(dest)
    marker = dest / ".extracted.json"
    if marker.exists():
        try:
            if json.loads(marker.read_text()).get("fingerprint") == fingerprint:
                log(f"Already extracted: {archive.name} -> {dest}")
                return dest
        except ValueError:
            pass
    tmp = dest.with_name(dest.name + ".tmp")
    shutil.rmtree(tmp, ignore_errors=True)
    shutil.rmtree(dest, ignore_errors=True)
    tmp.mkdir(parents=True)
    root = tmp.resolve()
    log(f"Extracting {archive.name} -> {dest} ...")
    try:
        if zipfile.is_zipfile(archive):
            with zipfile.ZipFile(archive) as zf:
                members = zf.infolist()
                need = sum(m.file_size for m in members)
                free = shutil.disk_usage(tmp).free
                if need > free * 0.95:
                    raise IOError(f"Not enough local disk: need {human(need)}, free {human(free)}.")
                for m in members:
                    target = (tmp / m.filename).resolve()
                    if target != root and root not in target.parents:
                        raise IOError(f"Unsafe path in archive (path traversal): {m.filename}")
                for i, m in enumerate(members, 1):
                    zf.extract(m, tmp)  # raises BadZipFile on CRC mismatch
                    if i % 20000 == 0:
                        print(f"    {i}/{len(members)} members")
        elif tarfile.is_tarfile(archive):
            with tarfile.open(archive) as tf:
                for m in tf.getmembers():
                    target = (tmp / m.name).resolve()
                    if target != root and root not in target.parents:
                        raise IOError(f"Unsafe path in archive (path traversal): {m.name}")
                    if m.issym() or m.islnk():
                        raise IOError(f"Archive contains links, refusing to extract: {m.name}")
                try:
                    tf.extractall(tmp, filter="data")
                except TypeError:  # Python < 3.12
                    tf.extractall(tmp)
        else:
            raise UserActionRequired(f"{archive.name} is not a zip or tar archive this notebook can read.")
    except (zipfile.BadZipFile, EOFError, tarfile.TarError) as e:
        shutil.rmtree(tmp, ignore_errors=True)
        raise IOError(f"{archive.name} is corrupt ({e}). Delete it and download it again.") from e
    (tmp / ".extracted.json").write_text(json.dumps({"archive": str(archive), "fingerprint": fingerprint,
                                                     "utc": datetime.now(timezone.utc).isoformat()}))
    tmp.replace(dest)
    return dest


def decode_check(paths, n, seed):
    """Fully decode a random sample of images; return the list of failures."""
    paths = list(paths)
    rng = random.Random(seed)
    bad = []
    for p in rng.sample(paths, min(n, len(paths))):
        try:
            with Image.open(p) as im:
                im.load()
        except Exception as e:
            bad.append(f"{p}: {e}")
    return bad


def frame_index(name):
    nums = re.findall(r"\d+", Path(name).stem)
    if not nums:
        raise ValueError(f"No frame number in file name {name}")
    return int(nums[-1])


def link_or_copy(src, dst):
    try:
        os.link(src, dst)
    except OSError:
        shutil.copy2(src, dst)


def clip_box(x1, y1, x2, y2, w, h):
    """Clip to the image; returns (box, was_clipped) or (None, reason) if invalid."""
    vals = (x1, y1, x2, y2)
    if not all(isinstance(v, (int, float)) and math.isfinite(v) for v in vals):
        return None, "non_finite"
    if x2 <= x1 or y2 <= y1:
        return None, "non_positive_size"
    cx1, cy1, cx2, cy2 = max(0.0, x1), max(0.0, y1), min(float(w), x2), min(float(h), y2)
    if cx2 - cx1 < 1 or cy2 - cy1 < 1:
        return None, "outside_or_tiny_after_clip"
    return (cx1, cy1, cx2, cy2), (cx1, cy1, cx2, cy2) != (x1, y1, x2, y2)


def yolo_line(cls_id, box, w, h):
    x1, y1, x2, y2 = box
    return (f"{cls_id} {(x1 + x2) / 2 / w:.6f} {(y1 + y2) / 2 / h:.6f} "
            f"{(x2 - x1) / w:.6f} {(y2 - y1) / h:.6f}")


def write_masked_image(src, dst, mask_boxes, keep_boxes, value):
    """Grey out mask_boxes, except pixels inside keep_boxes (labelled objects)."""
    with Image.open(src) as im:
        a = np.asarray(im.convert("RGB")).copy()
    h, w = a.shape[:2]
    m = np.zeros((h, w), dtype=bool)
    for x1, y1, x2, y2 in mask_boxes:
        m[max(0, int(math.floor(y1))):min(h, int(math.ceil(y2))),
          max(0, int(math.floor(x1))):min(w, int(math.ceil(x2)))] = True
    for x1, y1, x2, y2 in keep_boxes:
        m[max(0, int(math.floor(y1))):min(h, int(math.ceil(y2))),
          max(0, int(math.floor(x1))):min(w, int(math.ceil(x2)))] = False
    a[m] = value
    Image.fromarray(a).save(dst, quality=95)
    return float(m.mean())


def parse_policy(policy, classes):
    """'map:<cls>' / 'drop' / 'mask' / 'ignore_region' -> validated tuple."""
    out = {}
    for cat, rule in policy.items():
        if rule.startswith("map:"):
            cls = rule[4:]
            assert cls in classes, f"Policy maps '{cat}' to '{cls}', which is not in classes {classes}"
            out[cat] = ("map", classes.index(cls))
        elif rule in ("drop", "mask", "ignore_region"):
            out[cat] = (rule, None)
        else:
            raise ValueError(f"Unknown policy '{rule}' for category '{cat}'")
    return out


def reset_yolo_root(root):
    shutil.rmtree(root, ignore_errors=True)
    for split in ("train", "val", "test"):
        (root / "images" / split).mkdir(parents=True)
        (root / "labels" / split).mkdir(parents=True)


def write_dataset_yaml(root, names, variant):
    import yaml
    root = Path(root)
    for split in ("train", "val", "test"):
        assert (root / "images" / split).is_dir(), f"Missing {root / 'images' / split}"
    data = {"path": str(root), "train": "images/train", "val": "images/val", "test": "images/test",
            "names": {i: n for i, n in enumerate(names)}}
    assert len({data["train"], data["val"], data["test"]}) == 3, "train/val/test must be different folders"
    text = (f"# SmartFlow - {variant}. Generated by notebook 01; do not edit by hand.\n"
            "# Only images/train may be used for training or training-time augmentation.\n"
            + yaml.safe_dump(data, sort_keys=False))
    path = DIRS["local_yolo"] / f"{variant}.yaml"
    path.write_text(text)
    (DIRS["reports"] / f"{variant}.yaml").write_text(text)  # copy on Drive for the record
    reread = yaml.safe_load(path.read_text())
    assert reread["names"] == data["names"] and reread["path"] == str(root)
    log(f"Dataset YAML written: {path}")
    return path


def split_manifest(root):
    out = {}
    for split in ("train", "val", "test"):
        files = sorted(p.name for p in (Path(root) / "images" / split).iterdir()
                       if p.suffix.lower() in IMG_EXTS)
        out[split] = {"images": len(files),
                      "file_list_sha256": hashlib.sha256("\n".join(files).encode()).hexdigest()}
    return out


def validate_yolo_dataset(root, names, variant):
    """Independent re-check of the written YOLO dataset (reads files back from disk)."""
    root = Path(root)
    rep = Report(f"{variant}_validation")
    print(f"Validating YOLO dataset {variant} at {root}")
    stems, per_split = {}, {}
    for split in ("train", "val", "test"):
        imgs = {p.stem: p for p in (root / "images" / split).iterdir() if p.suffix.lower() in IMG_EXTS}
        lbls = {p.stem: p for p in (root / "labels" / split).glob("*.txt")}
        missing, orphan = sorted(imgs.keys() - lbls.keys()), sorted(lbls.keys() - imgs.keys())
        cls_counts, bad_lines, empty = Counter(), [], 0
        for stem, lp in lbls.items():
            lines = [l for l in lp.read_text().splitlines() if l.strip()]
            if not lines:
                empty += 1
            for ln in lines:
                parts = ln.split()
                try:
                    c, x, y, w, h = int(parts[0]), *map(float, parts[1:])
                    ok = (len(parts) == 5 and 0 <= c < len(names) and 0 <= x <= 1 and 0 <= y <= 1
                          and 0 < w <= 1 and 0 < h <= 1 and x - w / 2 >= -1e-6 and x + w / 2 <= 1 + 1e-6
                          and y - h / 2 >= -1e-6 and y + h / 2 <= 1 + 1e-6)
                except (ValueError, IndexError):
                    ok = False
                if ok:
                    cls_counts[names[c]] += 1
                else:
                    bad_lines.append(f"{lp.name}: {ln}")
        per_split[split] = {"images": len(imgs), "label_files": len(lbls), "empty_label_files": empty,
                            "boxes": sum(cls_counts.values()),
                            "boxes_per_class": {n: cls_counts.get(n, 0) for n in names}}
        stems[split] = set(imgs)
        print(f"  {split}: {per_split[split]}")
        rep.check(f"{split}: has images", len(imgs) > 0, len(imgs))
        rep.check(f"{split}: every image has a label file", not missing, missing[:5])
        rep.check(f"{split}: no orphan label files", not orphan, orphan[:5])
        rep.check(f"{split}: all label lines valid (class id, normalised xywh inside image)",
                  not bad_lines, bad_lines[:5])
        bad = decode_check(imgs.values(), CFG["sample_decode_per_split"], CFG["seed"])
        rep.check(f"{split}: sample of {min(CFG['sample_decode_per_split'], len(imgs))} images decodes",
                  not bad, bad[:5])
    for a, b in (("train", "val"), ("train", "test"), ("val", "test")):
        inter = stems[a] & stems[b]
        rep.check(f"no file-name overlap {a}/{b}", not inter, sorted(inter)[:5])
    if CFG["content_hash_leakage_check"]:
        seen, dups = {}, []
        for split in ("train", "val", "test"):
            for p in (root / "images" / split).iterdir():
                if p.suffix.lower() in IMG_EXTS:
                    d = hashlib.md5(p.read_bytes()).hexdigest()
                    if d in seen and seen[d][0] != split:
                        dups.append(f"{seen[d][1]} == {split}/{p.name}")
                    seen.setdefault(d, (split, f"{split}/{p.name}"))
        rep.check("no byte-identical images shared between splits", not dups, dups[:5])
    for n in names:
        rep.check(f"class '{n}' has training boxes", per_split["train"]["boxes_per_class"][n] > 0,
                  per_split["train"]["boxes_per_class"][n], fatal=False)
    rep.info.update(per_split=per_split, manifest=split_manifest(root))
    rep.finish()
    return rep


def show_samples(root, names, variant, split="train", n=12, seed=0):
    """Draw boxes read back from the YOLO label files (verifies the conversion, not the source)."""
    import matplotlib.pyplot as plt
    import matplotlib.patches as patches
    root = Path(root)
    imgs = sorted(p for p in (root / "images" / split).iterdir() if p.suffix.lower() in IMG_EXTS)
    labelled = [p for p in imgs if (root / "labels" / split / f"{p.stem}.txt").stat().st_size > 0]
    pick = random.Random(seed).sample(labelled or imgs, min(n, len(labelled or imgs)))
    cols = 4
    rows = max(1, math.ceil(len(pick) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 3.2 * rows))
    axes = np.atleast_1d(axes).ravel()
    colours = plt.get_cmap("tab10")
    for ax in axes:
        ax.axis("off")
    for ax, p in zip(axes, pick):
        with Image.open(p) as im:
            w, h = im.size
            ax.imshow(im.convert("RGB"))
        for ln in (root / "labels" / split / f"{p.stem}.txt").read_text().splitlines():
            c, x, y, bw, bh = ln.split()
            c, x, y, bw, bh = int(c), float(x) * w, float(y) * h, float(bw) * w, float(bh) * h
            ax.add_patch(patches.Rectangle((x - bw / 2, y - bh / 2), bw, bh, fill=False,
                                           lw=1.2, ec=colours(c % 10)))
            ax.text(x - bw / 2, y - bh / 2, names[c], fontsize=6, color="white",
                    bbox=dict(facecolor=colours(c % 10), alpha=0.7, pad=0.5, lw=0))
        ax.set_title(p.name[:48], fontsize=7)
    fig.suptitle(f"{variant} - {split} (boxes read back from YOLO labels)", fontsize=11)
    fig.tight_layout()
    out = DIRS["plots"] / f"{variant}_{split}_samples.png"
    fig.savefig(out, dpi=110)
    plt.show()
    plt.close(fig)
    log(f"Sample grid saved: {out}")


def mark_prepared(root, fingerprint, summary):
    (Path(root) / ".prepared.json").write_text(json.dumps({"fingerprint": fingerprint, "summary": summary},
                                                          default=str))


def persist_prepared(variant, root, fingerprint):
    if not CFG["persist_prepared_to_drive"]:
        return
    tar_path = DIRS["prepared_cache"] / f"{variant}.tar"
    man_path = DIRS["prepared_cache"] / f"{variant}.json"
    if man_path.exists() and tar_path.exists():
        try:
            if json.loads(man_path.read_text()).get("fingerprint") == fingerprint:
                log(f"Drive cache already up to date: {tar_path.name}")
                return
        except ValueError:
            pass
    tmp = tar_path.with_name(tar_path.name + ".tmp")
    need = sum(p.stat().st_size for p in Path(root).rglob("*") if p.is_file())
    free = shutil.disk_usage(DIRS["prepared_cache"]).free
    if need * 1.1 > free:
        raise UserActionRequired(
            f"Not enough Google Drive space to save {variant}: need ~{human(need)}, free {human(free)}. "
            "Free space (for example delete raw/ archives of datasets whose prepared_cache/*.tar already exists), "
            "then rerun this cell. Without this cache, notebook 02 cannot restore the dataset in a new session.")
    log(f"Writing prepared dataset cache to Drive: {tar_path} ({human(need)}) ...")
    with tarfile.open(tmp, "w") as tf:
        tf.add(root, arcname=variant)
    tmp.replace(tar_path)
    assert tar_path.stat().st_size >= need, f"{tar_path} is smaller than its contents - the Drive write failed."
    man_path.write_text(json.dumps({"fingerprint": fingerprint, "size": tar_path.stat().st_size,
                                    "sha256": file_hash(tar_path, "sha256")}))
    log(f"Cache written ({human(tar_path.stat().st_size)}).")


def try_restore_prepared(variant, root, fingerprint):
    """True if a validated dataset with this fingerprint is available locally or restorable from Drive."""
    root = Path(root)
    if CFG["force_rebuild"]:
        return False
    local = root / ".prepared.json"
    if local.exists():
        try:
            if json.loads(local.read_text()).get("fingerprint") == fingerprint:
                log(f"{variant}: already prepared locally with the current configuration.")
                return True
        except ValueError:
            pass
    tar_path = DIRS["prepared_cache"] / f"{variant}.tar"
    man_path = DIRS["prepared_cache"] / f"{variant}.json"
    if not (tar_path.exists() and man_path.exists()):
        return False
    man = json.loads(man_path.read_text())
    if man.get("fingerprint") != fingerprint:
        log(f"{variant}: Drive cache exists but was built with a different configuration -> rebuilding.")
        return False
    verify_file(tar_path, size=man["size"], sha256=man["sha256"])
    tmp = root.with_name(root.name + ".restore")
    shutil.rmtree(tmp, ignore_errors=True)
    extract_archive_once(tar_path, tmp, fingerprint=man["sha256"])
    shutil.rmtree(root, ignore_errors=True)
    (tmp / variant).replace(root)
    shutil.rmtree(tmp, ignore_errors=True)
    assert json.loads((root / ".prepared.json").read_text())["fingerprint"] == fingerprint
    log(f"{variant}: restored from Drive cache.")
    return True


def active(ds):
    """Should the raw-data cells for this dataset run?"""
    s = STATE.get(ds, {})
    if not CFG["enable"].get(ds):
        print(f"[{ds}] disabled in CFG['enable'] -> skipping.")
        return False
    if s.get("restored"):
        print(f"[{ds}] validated dataset already prepared (local or Drive cache) -> raw-data step skipped "
              "(set CFG['force_rebuild'] = True to redo it).")
        return False
    if s.get("status") == "USER_ACTION_REQUIRED":
        raise UserActionRequired(f"[{ds}] is waiting for you: {s.get('message')}")
    return True


print("Helpers loaded.")

## 7 · Dataset registry and exclusions
This records which datasets this notebook prepares and why the others are excluded. There is no code path that downloads or labels QMUL Junction or AI City 2021: they have no usable detection boxes, and labels are never fabricated.

In [ ]:
# 7. Dataset registry
assert "Report" in globals(), "Run cell 6 first."
REGISTRY = {
    "ua_detrac_mirror": {"variant": "ua_detrac_mirror_1cls", "classes": CFG["ua_detrac_mirror"]["classes"],
                         "acquisition": "automatic: public HF mirror, pinned revision, size+sha256 verified",
                         "licence_note": "mirror declares CC-BY-4.0 but is a third-party repackaging; "
                                         "authors' page gives no licence -> treat as research-only"},
    "bdd100k": {"variant": "bdd100k_4cls", "classes": CFG["bdd100k"]["classes"],
                "acquisition": "automatic from the BDD100K server at UC Berkeley (size/md5 verified)",
                "licence_note": "UC Berkeley licence: educational/research/not-for-profit use"},
    "ua_detrac_official": {"variant": "ua_detrac_official_4cls", "classes": CFG["ua_detrac_official"]["classes"],
                           "acquisition": "you download from the authors' Google Drive links (optional)",
                           "licence_note": "no licence text on the authors' page; cite the paper"},
}
EXCLUDED = {
    "qmul_junction": "single 360x288 video with no bounding boxes; its ground-truth zip holds clip-level "
                     "event labels + features (.mat) for unusual-event detection. Use only for "
                     "inference-only demos; detection metrics would need new manual annotation.",
    "aic2021": "access via the official request form; T1 = movement counts, T2 = ReID crops, "
               "T4 = anomaly timestamps (none are detection boxes); T3 boxes are class-agnostic and "
               "cover only vehicles seen by >=2 cameras -> unsuitable for detector training.",
}
for k, v in REGISTRY.items():
    print(f"[{'ENABLED ' if CFG['enable'][k] else 'disabled'}] {k:<20} -> {v['variant']}  classes={v['classes']}")
    print(f"{'':>32}acquisition: {v['acquisition']}\n{'':>32}licence: {v['licence_note']}")
for k, why in EXCLUDED.items():
    print(f"[EXCLUDED] {k:<20} {why}")
    STATE[k] = {"status": "EXCLUDED", "reason": why}

---
# Part A · UA-DETRAC (Hugging Face mirror) → `ua_detrac_mirror_1cls`
Verified facts this part relies on:
* Two archives, with sizes and sha256 values from the Hub API.
* Each archive holds one folder per sequence containing `annotations.csv` and the frames.
* The CSV columns are `filename,width,height,class,xmin,ymin,xmax,ymax,confidence,target_id`.
* The classes are `vehicle` and `ignored`, where `ignored` marks don't-care regions and is not a vehicle class.

### A1 · Acquire (download, resume, verify)
If a validated dataset with the same configuration already exists locally or in the Drive cache, it is restored and Part A's raw-data cells are skipped.

In [ ]:
# A1. UA-DETRAC mirror - acquire
DS = "ua_detrac_mirror"
C = CFG[DS]
VARIANT_A = REGISTRY[DS]["variant"]
YOLO_A = DIRS["local_yolo"] / VARIANT_A
FP_A = stable_hash({"cfg": C, "seed": CFG["seed"], "converter": CONVERTER_VERSION})
STATE[DS] = {"status": "PENDING", "fingerprint": FP_A}

if not CFG["enable"][DS]:
    STATE[DS]["status"] = "SKIPPED"
    print("UA-DETRAC mirror disabled in CFG['enable'].")
elif try_restore_prepared(VARIANT_A, YOLO_A, FP_A):
    STATE[DS]["restored"] = True
else:
    raw = DIRS["raw"] / DS
    raw.mkdir(parents=True, exist_ok=True)
    rep = Report(f"{DS}_00_acquisition")
    for fname, meta in C["files"].items():
        url = f"https://huggingface.co/datasets/{C['repo']}/resolve/{C['revision']}/{fname}"
        dest = download_resumable(url, raw / fname, expected_size=meta["size"])
        verify_file(dest, size=meta["size"], sha256=meta["sha256"])
        rep.check(f"{fname}: size and sha256 match the pinned revision", True, human(meta["size"]))
    rep.finish()

### A2 · Extract and verify structure
Each archive is extracted once to local disk. The cell then checks the number of sequence folders and images against the counts recorded in the audit, checks that every sequence has `annotations.csv`, checks that official train and test sequences don't overlap, and decodes a sample of images.

In [ ]:
# A2. UA-DETRAC mirror - extract + structure checks
if active("ua_detrac_mirror"):
    ext = DIRS["local_extracted"] / "ua_detrac_mirror"
    rep = Report("ua_detrac_mirror_01_structure")
    MIRROR_SEQS = {}
    for fname, meta in C["files"].items():
        split = meta["official_split"]
        out = extract_archive_once(DIRS["raw"] / "ua_detrac_mirror" / fname, ext / split, meta["sha256"])
        seq_dirs = sorted({p.parent for p in out.rglob("annotations.csv")})
        all_dirs = sorted({p.parent for p in out.rglob("*.jpg")})
        n_img = sum(1 for _ in out.rglob("*.jpg"))
        other = Counter(p.suffix.lower() for p in out.rglob("*") if p.is_file()
                        and p.suffix.lower() not in (".jpg", ".csv") and p.name != ".extracted.json")
        exp = C["expected"][split]
        rep.check(f"{split}: {exp['sequences']} sequence folders with annotations.csv",
                  len(seq_dirs) == exp["sequences"], len(seq_dirs))
        rep.check(f"{split}: every image folder has annotations.csv",
                  set(all_dirs) <= set(seq_dirs), sorted(str(d) for d in set(all_dirs) - set(seq_dirs))[:5])
        rep.check(f"{split}: {exp['images']} images", n_img == exp["images"], n_img)
        names = [d.name for d in seq_dirs]
        rep.check(f"{split}: sequence folder names unique", len(names) == len(set(names)))
        bad = decode_check([p for d in seq_dirs for p in d.glob("*.jpg")], CFG["sample_decode_per_split"],
                           CFG["seed"])
        rep.check(f"{split}: sample images decode", not bad, bad[:5])
        rep.info[f"{split}_other_files"] = dict(other)
        print(f"  {split}: other files present (not used): {dict(other)}")
        MIRROR_SEQS[split] = {d.name: d for d in seq_dirs}
    mvi = {s: {re.search(r"MVI_\d+", n).group(0) for n in MIRROR_SEQS[s]} for s in MIRROR_SEQS}
    rep.check("every sequence name contains an MVI id",
              all(re.search(r"MVI_\d+", n) for s in MIRROR_SEQS for n in MIRROR_SEQS[s]))
    rep.check("official train and test MVI ids are disjoint", not (mvi["train"] & mvi["test"]),
              sorted(mvi["train"] & mvi["test"]))
    rep.finish()

### A3 · Inspect annotations
For every sequence, the CSV is checked for its columns. The cell then counts images, rows per class, frames with and without vehicles, invalid boxes, CSV rows whose image is missing, and size mismatches against the real image header. **An unexpected class stops the notebook** rather than being dropped silently.

In [ ]:
# A3. UA-DETRAC mirror - annotation inspection
import pandas as pd
if active("ua_detrac_mirror"):
    assert "MIRROR_SEQS" in globals(), "Run cell A2 first."
    REQUIRED = ["filename", "width", "height", "class", "xmin", "ymin", "xmax", "ymax"]
    rep = Report("ua_detrac_mirror_02_annotations")
    MIRROR_ANN, rows_summary = {}, []
    unknown, missing_cols, missing_imgs, wh_mismatch = Counter(), [], [], []
    for split, seqs in MIRROR_SEQS.items():
        for seq, d in sorted(seqs.items()):
            df = pd.read_csv(d / "annotations.csv")
            miss = [c for c in REQUIRED if c not in df.columns]
            if miss:
                missing_cols.append(f"{seq}: {miss}")
                continue
            df["class"] = df["class"].astype(str)
            for c in set(df["class"]) - set(C["class_policy"]):
                unknown[c] += int((df["class"] == c).sum())
            on_disk = {p.name for p in d.glob("*.jpg")}
            in_csv = set(df["filename"])
            missing_imgs += [f"{seq}/{f}" for f in sorted(in_csv - on_disk)]
            veh = df[df["class"] == "vehicle"]
            coords = veh[["xmin", "ymin", "xmax", "ymax"]].astype(float)
            finite = np.isfinite(coords.values).all(axis=1)
            inval = (~finite) | (coords["xmax"] <= coords["xmin"]).values | (coords["ymax"] <= coords["ymin"]).values
            oob = ((coords["xmin"] < -1) | (coords["ymin"] < -1) | (coords["xmax"] > veh["width"] + 1)
                   | (coords["ymax"] > veh["height"] + 1)).values
            first = sorted(in_csv & on_disk)[:1]
            if first:
                with Image.open(d / first[0]) as im:
                    real = im.size
                row = df[df["filename"] == first[0]].iloc[0]
                if (int(row["width"]), int(row["height"])) != real:
                    wh_mismatch.append(f"{seq}: csv {(row['width'], row['height'])} vs image {real}")
            frames_with_veh = set(veh["filename"])
            rows_summary.append({"split": split, "sequence": seq, "images": len(on_disk),
                                 "vehicle_boxes": len(veh), "ignored_rows": int((df["class"] == "ignored").sum()),
                                 "frames_with_vehicles": len(frames_with_veh & on_disk),
                                 "frames_only_ignored_rows": len((in_csv - frames_with_veh) & on_disk),
                                 "frames_without_any_row": len(on_disk - in_csv),
                                 "invalid_boxes": int(inval.sum()), "out_of_bounds_boxes": int((oob & ~inval).sum())})
            MIRROR_ANN[(split, seq)] = df
    S = pd.DataFrame(rows_summary)
    tot = S.groupby("split")[[c for c in S.columns if c not in ("split", "sequence")]].sum()
    print(tot.to_string())
    rep.info["per_split"] = tot.to_dict(orient="index")
    rep.info["per_sequence"] = rows_summary
    rep.check("all CSVs have the required columns", not missing_cols, missing_cols[:5])
    rep.check("no annotation class without an explicit policy (nothing silently discarded)",
              not unknown, dict(unknown))
    rep.check("every CSV row refers to an image that exists", not missing_imgs,
              f"{len(missing_imgs)} missing, e.g. {missing_imgs[:3]}")
    rep.check("CSV width/height match the real image size", not wh_mismatch, wh_mismatch[:5])
    rep.check("invalid vehicle boxes (dropped + counted at conversion)", tot["invalid_boxes"].sum() == 0,
              int(tot["invalid_boxes"].sum()), fatal=False)
    rep.check("out-of-bounds vehicle boxes (clipped + counted at conversion)",
              tot["out_of_bounds_boxes"].sum() == 0, int(tot["out_of_bounds_boxes"].sum()), fatal=False)
    rep.check("frames without any CSV row "
              + ("(included as negatives)" if C["include_frames_without_rows"] else "(excluded, reported)"),
              tot["frames_without_any_row"].sum() == 0, int(tot["frames_without_any_row"].sum()), fatal=False)
    if unknown:
        STATE["ua_detrac_mirror"].update(status="USER_ACTION_REQUIRED",
                                         message=f"unexpected classes {dict(unknown)}")
        raise UserActionRequired(f"Unexpected annotation classes {dict(unknown)}. Add an explicit rule for each "
                                 "to CFG['ua_detrac_mirror']['class_policy'] and rerun from cell 1.")
    rep.finish()

### A4 · Leakage-free splits (by sequence, **before** frame sampling)
* **test** = all 40 official test sequences.
* **val** = `val_fraction` of the official train sequences, picked with the seed.
* **train** = the remaining sequences.

Frames are subsampled every `frame_stride` frames only after sequences are assigned, so neighbouring frames never land in different splits. The sequence lists are saved to `reports/ua_detrac_mirror_splits.json`. The official-XML part reuses that file so both UA-DETRAC variants share the same splits.

In [ ]:
# A4. UA-DETRAC mirror - sequence-level split, then frame selection
if active("ua_detrac_mirror"):
    assert "MIRROR_ANN" in globals(), "Run cell A3 first."
    rep = Report("ua_detrac_mirror_03_splits")
    rng = random.Random(CFG["seed"])
    official_train = sorted(MIRROR_SEQS["train"])
    shuffled = official_train[:]
    rng.shuffle(shuffled)
    n_val = max(1, round(len(official_train) * C["val_fraction"]))
    SPLIT_A = {"train": sorted(shuffled[n_val:]), "val": sorted(shuffled[:n_val]),
               "test": sorted(MIRROR_SEQS["test"])}
    mvi = {s: {re.search(r"MVI_\d+", n).group(0) for n in v} for s, v in SPLIT_A.items()}
    for a, b in (("train", "val"), ("train", "test"), ("val", "test")):
        rep.check(f"sequence sets disjoint: {a}/{b}", not (set(SPLIT_A[a]) & set(SPLIT_A[b])) and not (mvi[a] & mvi[b]))
    rep.check("train+val == official train sequences", sorted(SPLIT_A["train"] + SPLIT_A["val"]) == official_train)
    rep.check("val/test sequences never come from the other official split",
              set(SPLIT_A["val"]) <= set(MIRROR_SEQS["train"]) and set(SPLIT_A["test"]) <= set(MIRROR_SEQS["test"]))

    SELECTED_A = {}
    for split, seqs in SPLIT_A.items():
        src_split = "test" if split == "test" else "train"
        for seq in seqs:
            df = MIRROR_ANN[(src_split, seq)]
            frames = sorted((p.name for p in MIRROR_SEQS[src_split][seq].glob("*.jpg")), key=frame_index)
            if not C["include_frames_without_rows"]:
                with_rows = set(df["filename"])
                frames = [f for f in frames if f in with_rows]
            SELECTED_A[(split, seq)] = frames[::C["frame_stride"]]
    counts = {s: sum(len(v) for (sp, _), v in SELECTED_A.items() if sp == s) for s in SPLIT_A}
    print("sequences:", {s: len(v) for s, v in SPLIT_A.items()}, "\nselected frames:", counts)
    for s in SPLIT_A:
        rep.check(f"{s}: frames selected", counts[s] > 0, counts[s])
    (DIRS["reports"] / "ua_detrac_mirror_splits.json").write_text(json.dumps(
        {"seed": CFG["seed"], "val_fraction": C["val_fraction"], "frame_stride": C["frame_stride"],
         "sequences": SPLIT_A, "mvi_ids": {s: sorted(v) for s, v in mvi.items()},
         "selected_frames": counts}, indent=2))
    rep.info.update(sequences={s: len(v) for s, v in SPLIT_A.items()}, selected_frames=counts)
    rep.finish()

### A5 · Convert to YOLO
* `vehicle` rows become class 0. Boxes are clipped to the image, and invalid boxes are dropped and counted.
* `ignored` regions are filled with grey (114) **except** where they overlap a labelled vehicle. Unlabelled vehicles inside those regions would otherwise teach the model to treat real cars as background. Frames without ignored regions are hard-linked unchanged.
* Output names are `<sequence>__<frame>.jpg`, so every image–label pair is traceable to its source.
* An accounting assertion checks that every CSV row of every selected frame ends up in exactly one bucket.

In [ ]:
# A5. UA-DETRAC mirror - convert to YOLO
if active("ua_detrac_mirror"):
    assert "SELECTED_A" in globals(), "Run cell A4 first."
    rep = Report("ua_detrac_mirror_04_conversion")
    policy = parse_policy(C["class_policy"], C["classes"])
    reset_yolo_root(YOLO_A)
    st = Counter()
    written = Counter()
    for (split, seq), frames in SELECTED_A.items():
        src_split = "test" if split == "test" else "train"
        d = MIRROR_SEQS[src_split][seq]
        groups = {k: g for k, g in MIRROR_ANN[(src_split, seq)].groupby("filename")}
        for fname in frames:
            rows = groups.get(fname)
            with Image.open(d / fname) as im:
                w, h = im.size
            lines, keep, mask = [], [], []
            n_rows = 0 if rows is None else len(rows)
            records = [] if rows is None else zip(rows["class"], rows["xmin"], rows["ymin"],
                                                  rows["xmax"], rows["ymax"])
            for cls_name, x1, y1, x2, y2 in records:
                kind, cls_id = policy[str(cls_name)]
                box, info = clip_box(float(x1), float(y1), float(x2), float(y2), w, h)
                if box is None:
                    st[f"dropped_invalid_{kind}:{info}"] += 1
                    continue
                if kind == "map":
                    lines.append(yolo_line(cls_id, box, w, h))
                    keep.append(box)
                    st["boxes_written"] += 1
                    st["boxes_clipped"] += int(info)
                elif kind == "ignore_region":
                    mask.append(box)
                    st["ignore_regions_masked"] += 1
                else:
                    raise AssertionError(f"policy '{kind}' not supported for this dataset")
            accounted = (sum(v for k, v in st.items() if k.startswith("dropped_invalid"))
                         + st["boxes_written"] + st["ignore_regions_masked"])
            st["rows_seen"] += n_rows
            assert accounted == st["rows_seen"], f"row accounting broken at {seq}/{fname}"
            stem = f"{seq}__{Path(fname).stem}"
            dst = YOLO_A / "images" / split / f"{stem}.jpg"
            if mask:
                st["images_masked"] += 1
                write_masked_image(d / fname, dst, mask, keep, C["mask_value"])
            else:
                link_or_copy(d / fname, dst)
            (YOLO_A / "labels" / split / f"{stem}.txt").write_text("\n".join(lines) + ("\n" if lines else ""))
            st["empty_label_files"] += int(not lines)
            written[split] += 1
    expected = {s: sum(len(v) for (sp, _), v in SELECTED_A.items() if sp == s) for s in ("train", "val", "test")}
    print(dict(st), "\nwritten:", dict(written))
    rep.info.update(stats=dict(st), written=dict(written))
    for s in expected:
        rep.check(f"{s}: images written == frames selected", written[s] == expected[s], f"{written[s]}/{expected[s]}")
    rep.check("every annotation row accounted for (written / masked / dropped-invalid)", True, st["rows_seen"])
    rep.check("invalid boxes dropped", not any(k.startswith("dropped_invalid") for k in st),
              {k: v for k, v in st.items() if k.startswith("dropped_invalid")}, fatal=False)
    rep.finish()

### A6 · Dataset YAML, independent validation, sample grid
This always runs, including after a cache restore. The validation re-reads the written files and checks:
* every image has a label file, and there are no orphan labels;
* every label line is valid (class id in range, box normalised and inside the image);
* a sample of images decodes;
* no file names or **byte-identical images** are shared across splits;
* per-class box counts.

The grid draws boxes **read back from the YOLO labels**, so it tests the conversion itself. Once validation passes, the dataset is cached to Drive.

In [ ]:
# A6. UA-DETRAC mirror - YAML + validation + visualisation
if CFG["enable"]["ua_detrac_mirror"] and STATE.get("ua_detrac_mirror", {}).get("status") in ("PENDING", "PASS"):
    names = C["classes"]
    yaml_a = write_dataset_yaml(YOLO_A, names, VARIANT_A)
    vrep = validate_yolo_dataset(YOLO_A, names, VARIANT_A)
    mark_prepared(YOLO_A, FP_A, vrep.info["per_split"])
    persist_prepared(VARIANT_A, YOLO_A, FP_A)
    STATE["ua_detrac_mirror"].update(status="PASS", yaml=str(yaml_a), variant=VARIANT_A,
                                     per_split=vrep.info["per_split"], manifest=vrep.info["manifest"])
    for split in ("train", "val", "test"):
        show_samples(YOLO_A, names, VARIANT_A, split=split, n=8, seed=CFG["seed"])

---
# Part B · BDD100K → `bdd100k_4cls`
### B1 · Acquire (automatic download)
**Nothing to do by hand.** The cell downloads two files from the BDD100K server at UC Berkeley (`dl.yf.io`, run by the dataset's lead author). The download resumes if interrupted and is saved to `<drive_root>/raw/bdd100k/`, so it happens only once:

| File | Size | Contents (checked 2026-10-21) | Integrity check |
|---|---|---|---|
| `bdd100k_images_100k.zip` | 5.67 GB | 100,000 images: 70,000 train / 10,000 val / 20,000 test | exact size, CRC of every file on extraction, exact image counts, every labelled image present |
| `bdd100k_labels.zip` | 190 MB | official **2018** detection labels, one JSON per image | exact size + md5 |

Some things to know before you start:
* **Licence.** The BDD100K licence (UC Regents) allows educational, research and not-for-profit use. The cell downloads `LICENSE.txt` next to the data and prints it, and by downloading you accept it.
* **Why the 2018 labels?** The docs recommend the newer "Detection 2020 Labels", but the ETH server that hosts them (`dl.cv.ethz.ch`) has been unreachable since March 2026. The Berkeley server has only the 2018 set. Its `bdd100k_det_20_labels.zip`, despite the name, contains test images, not labels, so it is not used. The 2018 labels are the official original release, and the 2020 set is a re-checked revision of the same images. The label version is recorded in every report and in the data gate, so your results state which one was used.
* **If you ever obtain the official 2020 labels**, drop the zip into `raw/bdd100k/`. It is recognised by its official md5 and used automatically instead of the 2018 set.

If the server is unreachable, the cell stops with `UserActionRequired` and explains your options. It never uses unofficial copies. To continue without BDD100K, set `CFG["enable"]["bdd100k"] = False`.

In [ ]:
# B1. BDD100K - automatic download / identification
import requests
DS = "bdd100k"
CB = CFG[DS]
VARIANT_B = REGISTRY[DS]["variant"]
YOLO_B = DIRS["local_yolo"] / VARIANT_B
STATE[DS] = {"status": "PENDING"}
NAMES_B = {"images_100k": "100K Images", "det_20_labels": "Detection 2020 Labels (official, preferred)",
           "labels_2018": "Detection labels 2018 (official, per-image JSON)"}


def identify_bdd_archives(raw):
    # Recognise archives by md5 (official docs + pinned), or by exact name+size when no md5 is pinned.
    by_md5 = {m["md5"]: k for k, m in CB["official_archives"].items()}
    by_md5.update({m["md5"]: k for k, m in CB["auto_source"]["files"].items() if m["md5"]})
    by_name = {m["file"]: (k, m) for k, m in CB["auto_source"]["files"].items() if not m["md5"]}
    found, unmatched = {}, []
    for z in sorted(raw.glob("*.zip")):
        if z.name in by_name and z.stat().st_size == by_name[z.name][1]["size"]:
            found.setdefault(by_name[z.name][0], z)
            continue
        key = by_md5.get(file_hash(z, "md5"))
        if key:
            found.setdefault(key, z)
        else:
            unmatched.append(z.name)
    return found, unmatched


def reachable_base():
    for base in CB["auto_source"]["base_urls"]:
        try:
            r = requests.head(base + "LICENSE.txt", timeout=15, allow_redirects=True)
            if r.status_code == 200:
                return base
            log(f"{base}: HTTP {r.status_code}")
        except requests.RequestException as e:
            log(f"{base}: unreachable ({e.__class__.__name__})")
    return None


if not CFG["enable"][DS]:
    STATE[DS]["status"] = "SKIPPED"
    print("BDD100K disabled in CFG['enable'].")
else:
    raw = DIRS["raw"] / DS
    raw.mkdir(parents=True, exist_ok=True)
    BDD_ARCHIVES, unmatched = identify_bdd_archives(raw)
    has_labels = "det_20_labels" in BDD_ARCHIVES or "labels_2018" in BDD_ARCHIVES
    need = [k for k in ("images_100k",) if k not in BDD_ARCHIVES] + ([] if has_labels else ["labels_2018"])
    if need and CB["auto_download"]:
        base = reachable_base()
        if base is None:
            msg = ("The BDD100K servers are unreachable right now: " + ", ".join(CB["auto_source"]["base_urls"])
                   + ". Options: (1) rerun this cell later - nothing already downloaded is lost; "
                   "(2) if you have the official BDD100K zips, upload them unchanged to " + str(raw)
                   + " (they are recognised by md5); (3) set CFG['enable']['bdd100k'] = False to continue "
                   "with UA-DETRAC only.")
            STATE[DS].update(status="USER_ACTION_REQUIRED", message=msg)
            raise UserActionRequired(msg)
        lic = download_resumable(base + "LICENSE.txt", raw / "LICENSE.txt")
        print("\n" + lic.read_text()[:700] + " ...\n(full text: " + str(lic) + ")\n")
        for key in need:
            meta = CB["auto_source"]["files"][key]
            dest = download_resumable(base + meta["file"], raw / meta["file"], expected_size=meta["size"])
            verify_file(dest, size=meta["size"], md5=meta["md5"])
        BDD_ARCHIVES, unmatched = identify_bdd_archives(raw)
    for name in unmatched:
        log(f"Ignoring {name}: it matches no known BDD100K package (md5 or name+size).")
    BDD_LABEL_KIND = "det_20_labels" if "det_20_labels" in BDD_ARCHIVES else (
        "labels_2018" if "labels_2018" in BDD_ARCHIVES else None)
    missing = ([] if "images_100k" in BDD_ARCHIVES else ["100K Images"]) + ([] if BDD_LABEL_KIND else ["labels"])
    if missing:
        msg = (f"BDD100K files still missing in {raw}: {missing}. Set CFG['bdd100k']['auto_download'] = True "
               "and rerun, or upload the official zips unchanged to that folder.")
        STATE[DS].update(status="USER_ACTION_REQUIRED", message=msg)
        raise UserActionRequired(msg)
    used = {k: v for k, v in BDD_ARCHIVES.items() if k in ("images_100k", BDD_LABEL_KIND)}
    provenance = {k: {"file": v.name, "size": v.stat().st_size, "sha256": file_hash(v, "sha256")}
                  for k, v in used.items()}
    (raw / "provenance.json").write_text(json.dumps(provenance, indent=2))
    FP_B = stable_hash({"cfg": {k: v for k, v in CB.items() if k != "auto_source"}, "archives": provenance,
                        "seed": CFG["seed"], "converter": CONVERTER_VERSION})
    STATE[DS].update(fingerprint=FP_B, labels=BDD_LABEL_KIND, archives=provenance)
    rep = Report("bdd100k_00_acquisition")
    for k, v in used.items():
        rep.check(f"{NAMES_B[k]}: present and verified", True, v.name)
    rep.check("using the 2020 detection labels recommended by the docs", BDD_LABEL_KIND == "det_20_labels",
              f"using {NAMES_B[BDD_LABEL_KIND]}", fatal=False)
    rep.info["provenance"] = provenance
    rep.finish()
    if try_restore_prepared(VARIANT_B, YOLO_B, FP_B):
        STATE[DS]["restored"] = True

### B2 · Extract and verify structure
The cell extracts both archives once, then searches for the `100k/{train,val,test}` image folders and for the label files, rather than assuming a top-level folder. The label files are either `det_{train,val}.json` (2020) or `100k/{train,val}/*.json` (2018). Image counts must be exactly 70,000 / 10,000 / 20,000.

In [ ]:
# B2. BDD100K - extract + structure checks
if active("bdd100k"):
    assert "BDD_ARCHIVES" in globals(), "Run cell B1 first."
    ext = DIRS["local_extracted"] / "bdd100k"
    rep = Report("bdd100k_01_structure")
    for key in ("images_100k", BDD_LABEL_KIND):
        extract_archive_once(BDD_ARCHIVES[key], ext / key, STATE["bdd100k"]["archives"][key]["sha256"])
    roots = [p for p in (ext / "images_100k").rglob("100k") if p.is_dir() and (p / "train").is_dir()]
    rep.check("exactly one 100k/ image folder with train/ found", len(roots) == 1, [str(r) for r in roots])
    BDD_IMG_ROOT = roots[0]
    BDD_LABELS = {}
    for split in ("train", "val"):
        if BDD_LABEL_KIND == "det_20_labels":
            hits = list((ext / BDD_LABEL_KIND).rglob(f"det_{split}.json"))
        else:
            hits = [p for p in (ext / BDD_LABEL_KIND).rglob(split)
                    if p.is_dir() and p.parent.name == "100k" and any(p.glob("*.json"))]
        rep.check(f"{split}: exactly one label location found", len(hits) == 1, [str(h) for h in hits])
        BDD_LABELS[split] = hits[0]
    BDD_IMAGES = {}
    for split in ("train", "val", "test"):
        files = {p.name: p for p in (BDD_IMG_ROOT / split).glob("*") if p.suffix.lower() in IMG_EXTS} \
            if (BDD_IMG_ROOT / split).is_dir() else {}
        BDD_IMAGES[split] = files
        exp = CB["expected_images"][split]
        rep.check(f"{split}: exactly {exp} images", len(files) == exp, len(files))
        if split != "test":
            bad = decode_check(files.values(), CFG["sample_decode_per_split"], CFG["seed"])
            rep.check(f"{split}: sample images decode", not bad, bad[:5])
    print("Images root:", BDD_IMG_ROOT, "\nLabels:", BDD_LABELS, "\nLabel version:", NAMES_B[BDD_LABEL_KIND])
    print("Note: official test images are not used (test = official val, see B4).")
    rep.finish()

### B3 · Inspect annotations
Both label versions are first converted into the same in-memory form: one record per image, holding its name and its labels (`category`, `box2d`). The 2018 per-image files must have `name` equal to the file name and exactly one frame. Then, for each official split, the cell counts:
* image names that are duplicated;
* labelled images that are missing on disk;
* images on disk that have no label entry, which are excluded and reported;
* frames with no labels;
* labels without `box2d`;
* invalid and out-of-bounds boxes;
* boxes per category.

Image sizes come from the real image headers. **Every category found must have an explicit rule in `category_policy`**; otherwise the notebook stops and lists the categories.

In [ ]:
# B3. BDD100K - annotation inspection
import pandas as pd
if active("bdd100k"):
    assert "BDD_LABELS" in globals(), "Run cell B2 first."
    rep = Report("bdd100k_02_annotations")

    def load_frames(path):
        if BDD_LABEL_KIND == "det_20_labels":
            with open(path) as f:
                data = json.load(f)
            frames = data["frames"] if isinstance(data, dict) and "frames" in data else data
            assert isinstance(frames, list), f"{path}: expected a list of frames, got {type(frames).__name__}"
            return frames, []
        frames, problems = [], []
        for p in sorted(Path(path).glob("*.json")):
            d = json.loads(p.read_text())
            if d.get("name") != p.stem:
                problems.append(f"{p.name}: name field {d.get('name')!r}")
            frs = d.get("frames") or []
            if len(frs) != 1:
                problems.append(f"{p.name}: {len(frs)} frames (expected 1)")
            labels = [{"category": o.get("category"), **({"box2d": o["box2d"]} if o.get("box2d") else {})}
                      for f in frs for o in (f.get("objects") or [])]
            frames.append({"name": p.stem + ".jpg", "labels": labels})
        return frames, problems

    BDD_FRAMES, BDD_SIZES = {}, {}
    cat_counts, rows = defaultdict(Counter), []
    for split in ("train", "val"):
        frames, problems = load_frames(BDD_LABELS[split])
        rep.check(f"{split}: per-image label files well-formed", not problems, problems[:5])
        schema_bad = [i for i, fr in enumerate(frames[:1000]) if not isinstance(fr, dict) or "name" not in fr]
        rep.check(f"{split}: frames have a 'name' field", not schema_bad, schema_bad[:5])
        names = [fr["name"] for fr in frames]
        dup = [n for n, c in Counter(names).items() if c > 1]
        rep.check(f"{split}: image names unique in label file", not dup, dup[:5])
        on_disk = BDD_IMAGES[split]
        missing = sorted(set(names) - set(on_disk))
        unlabeled = sorted(set(on_disk) - set(names))
        rep.check(f"{split}: every labelled image exists on disk", not missing or CB["allow_missing_images"],
                  f"{len(missing)} missing, e.g. {missing[:3]}")
        rep.check(f"{split}: images on disk without a label entry (excluded, reported)", not unlabeled,
                  len(unlabeled), fatal=False)
        sizes, unreadable = {}, []
        for n in names:
            if n in on_disk:
                try:
                    with Image.open(on_disk[n]) as im:
                        sizes[n] = im.size
                except Exception as e:
                    unreadable.append(f"{n}: {e}")
        rep.check(f"{split}: all labelled image headers readable", not unreadable, unreadable[:5])
        st = Counter()
        for fr in frames:
            labels = fr.get("labels") or []
            st["frames"] += 1
            st["frames_without_labels"] += int(not labels)
            wh = sizes.get(fr["name"])
            for lab in labels:
                cat = lab.get("category", "<missing category>")
                box = lab.get("box2d")
                if not box:
                    cat_counts[split][f"{cat} (no box2d)"] += 1
                    st["labels_without_box2d"] += 1
                    continue
                cat_counts[split][cat] += 1
                try:
                    x1, y1, x2, y2 = (float(box[k]) for k in ("x1", "y1", "x2", "y2"))
                except (KeyError, TypeError, ValueError):
                    st["malformed_box2d"] += 1
                    continue
                if wh is None:
                    continue
                res, info = clip_box(x1, y1, x2, y2, *wh)
                if res is None:
                    st[f"invalid:{info}"] += 1
                elif info:
                    st["out_of_bounds_clipped"] += 1
        rows.append({"split": split, **st, "images_on_disk": len(on_disk), "labelled_missing_on_disk": len(missing),
                     "on_disk_without_label_entry": len(unlabeled)})
        rep.check(f"{split}: no malformed box2d", st["malformed_box2d"] == 0, st["malformed_box2d"])
        BDD_FRAMES[split] = [fr for fr in frames if fr["name"] in sizes]
        BDD_SIZES.update(sizes)
        size_hist = Counter(sizes.values())
        print(f"  {split}: image sizes {dict(size_hist.most_common(5))}")
    print(pd.DataFrame(rows).fillna(0).to_string(index=False))
    cats = sorted({c for s in cat_counts for c in cat_counts[s]})
    table = pd.DataFrame({"category": cats,
                          "policy": ["not a box -> excluded" if c.endswith("(no box2d)")
                                     else CB["category_policy"].get(c, "<NO POLICY>") for c in cats],
                          **{s: [cat_counts[s].get(c, 0) for c in cats] for s in ("train", "val")}})
    print("\n" + table.to_string(index=False))
    unknown = sorted({c for c in cats if "(no box2d)" not in c} - set(CB["category_policy"]))
    never = sorted(set(CB["category_policy"]) - set(cats))
    rep.info.update(label_version=BDD_LABEL_KIND, per_split=rows, categories=table.to_dict(orient="records"),
                    policy_never_observed=never)
    rep.check("labels without box2d (not detection boxes; excluded, counted)",
              not any("(no box2d)" in c for c in cats), [c for c in cats if "(no box2d)" in c], fatal=False)
    rep.check("policy entries never observed in the data (informational)", not never, never, fatal=False)
    rep.check("every observed category has an explicit policy (nothing silently discarded)", not unknown, unknown)
    if unknown:
        STATE["bdd100k"].update(status="USER_ACTION_REQUIRED", message=f"categories without policy: {unknown}")
        raise UserActionRequired(f"Categories without a rule: {unknown}. Add each to "
                                 "CFG['bdd100k']['category_policy'] as 'map:<class>', 'drop' or 'mask', then rerun "
                                 "from cell 1.")
    rep.finish()

### B4 · Leakage-free splits
* **test** = the official `val` split. Official test labels are not public.
* **val** = `val_fraction` of the official `train` images, picked with the seed.
* **train** = the rest.

Per the docs, each 100K image is the frame at the 10th second of a distinct video, so an image-level split here is also a video-level split. Names are checked for overlap now, and file contents are hashed in B6.

In [ ]:
# B4. BDD100K - splits
if active("bdd100k"):
    assert "BDD_FRAMES" in globals(), "Run cell B3 first."
    rep = Report("bdd100k_03_splits")
    official_train = sorted(fr["name"] for fr in BDD_FRAMES["train"])
    shuffled = official_train[:]
    random.Random(CFG["seed"]).shuffle(shuffled)
    n_val = max(1, round(len(shuffled) * CB["val_fraction"]))
    SPLIT_B = {"train": sorted(shuffled[n_val:]), "val": sorted(shuffled[:n_val]),
               "test": sorted(fr["name"] for fr in BDD_FRAMES["val"])}
    for a, b in (("train", "val"), ("train", "test"), ("val", "test")):
        inter = set(SPLIT_B[a]) & set(SPLIT_B[b])
        rep.check(f"image names disjoint: {a}/{b}", not inter, sorted(inter)[:5])
    rep.check("test comes only from official val", set(SPLIT_B["test"]) <= set(BDD_IMAGES["val"]))
    rep.check("train/val come only from official train",
              set(SPLIT_B["train"]) | set(SPLIT_B["val"]) <= set(BDD_IMAGES["train"]))
    counts = {s: len(v) for s, v in SPLIT_B.items()}
    print(counts)
    (DIRS["reports"] / "bdd100k_splits.json").write_text(json.dumps(
        {"seed": CFG["seed"], "val_fraction": CB["val_fraction"], "counts": counts,
         "val_names": SPLIT_B["val"]}, indent=2))
    rep.info["counts"] = counts
    rep.finish()

### B5 · Convert to YOLO
Each `category_policy` rule works as follows:
* **`map`** writes a box with that class id. The class order is `CFG["bdd100k"]["classes"]`.
* **`drop`** marks a category that isn't a SmartFlow target, such as pedestrians or traffic signs. It is removed and counted.
* **`mask`** marks a vehicle-like category that is excluded. Its region is greyed out (except where it overlaps mapped boxes), so it can't act as an unlabelled vehicle.

An accounting assertion checks that every box-bearing label lands in exactly one bucket. Unmodified images are hard-linked.

In [ ]:
# B5. BDD100K - convert to YOLO
if active("bdd100k"):
    assert "SPLIT_B" in globals(), "Run cell B4 first."
    rep = Report("bdd100k_04_conversion")
    classes = CB["classes"]
    policy = parse_policy(CB["category_policy"], classes)
    reset_yolo_root(YOLO_B)
    by_name = {fr["name"]: fr for s in ("train", "val") for fr in BDD_FRAMES[s]}
    src_dir = {"train": BDD_IMAGES["train"], "val": BDD_IMAGES["train"], "test": BDD_IMAGES["val"]}
    st, per_cat, written = Counter(), defaultdict(Counter), Counter()
    for split, names in SPLIT_B.items():
        for i, name in enumerate(names, 1):
            fr = by_name[name]
            w, h = BDD_SIZES[name]
            lines, keep, mask = [], [], []
            for lab in fr.get("labels") or []:
                cat, box = lab.get("category"), lab.get("box2d")
                if not box:
                    per_cat[cat]["no_box2d"] += 1
                    continue
                st["box_labels_seen"] += 1
                kind, cls_id = policy[cat]
                if kind == "drop":
                    per_cat[cat]["dropped_by_policy"] += 1
                    continue
                res, info = clip_box(float(box["x1"]), float(box["y1"]), float(box["x2"]), float(box["y2"]), w, h)
                if res is None:
                    per_cat[cat][f"dropped_invalid:{info}"] += 1
                    continue
                if kind == "map":
                    lines.append(yolo_line(cls_id, res, w, h))
                    keep.append(res)
                    per_cat[cat]["written_as_" + classes[cls_id]] += 1
                    st["boxes_clipped"] += int(info)
                elif kind == "mask":
                    mask.append(res)
                    per_cat[cat]["masked"] += 1
            dst = YOLO_B / "images" / split / name
            if mask:
                st["images_masked"] += 1
                write_masked_image(src_dir[split][name], dst, mask, keep, CB["mask_value"])
            else:
                link_or_copy(src_dir[split][name], dst)
            (YOLO_B / "labels" / split / f"{Path(name).stem}.txt").write_text("\n".join(lines) + ("\n" if lines else ""))
            st["empty_label_files"] += int(not lines)
            written[split] += 1
            if i % 20000 == 0:
                print(f"    {split}: {i}/{len(names)}")
    accounted = sum(v for c in per_cat.values() for k, v in c.items() if k != "no_box2d")
    print(pd.DataFrame(per_cat).T.fillna(0).astype(int).to_string())
    print(dict(st), "\nwritten:", dict(written))
    rep.info.update(stats=dict(st), per_category={k: dict(v) for k, v in per_cat.items()}, written=dict(written))
    rep.check("every box label accounted for (written / masked / dropped by policy / dropped invalid)",
              accounted == st["box_labels_seen"], f"{accounted}/{st['box_labels_seen']}")
    for s in SPLIT_B:
        rep.check(f"{s}: images written == split size", written[s] == len(SPLIT_B[s]), f"{written[s]}/{len(SPLIT_B[s])}")
    rep.finish()

### B6 · Dataset YAML, independent validation, sample grid

In [ ]:
# B6. BDD100K - YAML + validation + visualisation
if CFG["enable"]["bdd100k"] and STATE.get("bdd100k", {}).get("status") in ("PENDING", "PASS"):
    names = CB["classes"]
    yaml_b = write_dataset_yaml(YOLO_B, names, VARIANT_B)
    vrep = validate_yolo_dataset(YOLO_B, names, VARIANT_B)
    mark_prepared(YOLO_B, FP_B, vrep.info["per_split"])
    persist_prepared(VARIANT_B, YOLO_B, FP_B)
    STATE["bdd100k"].update(status="PASS", yaml=str(yaml_b), variant=VARIANT_B,
                            per_split=vrep.info["per_split"], manifest=vrep.info["manifest"])
    for split in ("train", "val", "test"):
        show_samples(YOLO_B, names, VARIANT_B, split=split, n=8, seed=CFG["seed"])

---
# Part C · UA-DETRAC official annotations → `ua_detrac_official_4cls` (optional)
This part is disabled by default; enable it with `CFG["enable"]["ua_detrac_official"] = True`. It needs files only you can download:

1. Open the UA-DETRAC authors' page `https://sites.google.com/view/daweidu/projects/ua-detrac`.
2. Follow its **DETRAC-Images** and **DETRAC-Annos** Google Drive links. Google may ask you to sign in, and you should check the terms shown there.
3. Upload the downloaded archives, unchanged, into `<drive_root>/raw/ua_detrac_official/`.

No checksums are published for these files. On first use the notebook records their sha256 in `provenance.json`, and on later runs it warns if they change.

**Schema check:** the expected XML structure is assumed, not verified in this audit. It is `<sequence name>` → `<ignored_region><box …/>` and `<frame num>` → `<target_list><target>` → `<box left top width height>` + `<attribute vehicle_type>`. The cells check every file against this structure and stop with an explanation if it differs. Nothing is converted unless every check passes.

The official train/test membership and our train/val assignment are taken from `reports/ua_detrac_mirror_splits.json`, so run Part A first. That way both UA-DETRAC variants use **identical** sequence splits.

In [ ]:
# C1. UA-DETRAC official - acquire, extract, locate XML + image folders
import xml.etree.ElementTree as ET
DS = "ua_detrac_official"
CO = CFG[DS]
VARIANT_C = REGISTRY[DS]["variant"]
YOLO_C = DIRS["local_yolo"] / VARIANT_C
if not CFG["enable"][DS]:
    STATE[DS] = {"status": "SKIPPED"}
    print("UA-DETRAC official annotations disabled in CFG['enable'] (optional).")
else:
    split_file = DIRS["reports"] / "ua_detrac_mirror_splits.json"
    if not split_file.exists():
        raise UserActionRequired("Run Part A (UA-DETRAC mirror) first: Part C reuses its sequence splits "
                                 f"from {split_file}.")
    SPLITS_FROM_A = json.loads(split_file.read_text())
    raw = DIRS["raw"] / DS
    raw.mkdir(parents=True, exist_ok=True)
    archives = sorted(p for p in raw.iterdir() if p.is_file() and not p.name.endswith((".json", ".part"))
                      and (zipfile.is_zipfile(p) or tarfile.is_tarfile(p)))
    prov_path = raw / "provenance.json"
    prov = json.loads(prov_path.read_text()) if prov_path.exists() else {}
    current = {p.name: file_hash(p, "sha256") for p in archives}
    FP_C = stable_hash({"cfg": CO, "archives": current, "splits": SPLITS_FROM_A, "converter": CONVERTER_VERSION})
    STATE[DS] = {"status": "PENDING", "fingerprint": FP_C}
    if archives and try_restore_prepared(VARIANT_C, YOLO_C, FP_C):
        STATE[DS]["restored"] = True
    elif not archives:
        msg = (f"No archives found in {raw}. Download 'DETRAC-Images' and 'DETRAC-Annos' via the Google Drive "
               "links on https://sites.google.com/view/daweidu/projects/ua-detrac and upload them unchanged "
               "to that folder (zip or tar). Or set CFG['enable']['ua_detrac_official'] = False.")
        STATE[DS].update(status="USER_ACTION_REQUIRED", message=msg)
        raise UserActionRequired(msg)
    else:
        rep = Report("ua_detrac_official_01_structure")
        changed = {k: (prov[k], v) for k, v in current.items() if k in prov and prov[k] != v}
        rep.check("archives unchanged since first recorded (provenance.json)", not changed, changed, fatal=False)
        prov.update(current)
        prov_path.write_text(json.dumps(prov, indent=2))
        ext = DIRS["local_extracted"] / DS
        for p in archives:
            extract_archive_once(p, ext / p.stem, current[p.name])
        XML_FILES, dup_seq = {}, []
        for x in ext.rglob("*.xml"):
            try:
                root_tag = next(ET.iterparse(x, events=("start",)))[1].tag
            except ET.ParseError:
                continue
            if root_tag == "sequence":
                seq = ET.parse(x).getroot().get("name") or x.stem
                if seq in XML_FILES:
                    dup_seq.append(f"{seq}: {XML_FILES[seq]} and {x}")
                XML_FILES[seq] = x
        IMG_DIRS = defaultdict(list)
        for d in {p.parent for p in ext.rglob("*.jpg")}:
            IMG_DIRS[d.name].append(d)
        dup_dirs = {k: [str(x) for x in v] for k, v in IMG_DIRS.items() if len(v) > 1}
        rep.check("found <sequence> XML annotation files", len(XML_FILES) > 0, len(XML_FILES))
        rep.check("each sequence annotated by exactly one XML file", not dup_seq, dup_seq[:5])
        rep.check("image folder names are unique", not dup_dirs, list(dup_dirs)[:5])
        no_imgs = sorted(s for s in XML_FILES if s not in IMG_DIRS)
        rep.check("every annotated sequence has an image folder of the same name", not no_imgs, no_imgs[:5])
        known = {re.search(r"MVI_\d+", n).group(0): s for s, v in SPLITS_FROM_A["sequences"].items() for n in v}
        unknown_seq = sorted(s for s in XML_FILES if s not in known)
        rep.check("every XML sequence is in the mirror's official train/test lists", not unknown_seq, unknown_seq[:5])
        cover = Counter(known[s] for s in XML_FILES if s in known)
        rep.check("annotated sequences cover train, val and test", all(cover[s] > 0 for s in ("train", "val", "test")),
                  dict(cover))
        SEQ_SPLIT_C = {s: known[s] for s in XML_FILES if s in known}
        rep.finish()

### C2 · Inspect official XML annotations (schema-validated)

In [ ]:
# C2. UA-DETRAC official - schema validation + inspection
if active("ua_detrac_official"):
    assert "XML_FILES" in globals(), "Run cell C1 first."
    rep = Report("ua_detrac_official_02_annotations")
    OFFICIAL_ANN, schema_errors, vtypes, rows = {}, [], Counter(), []
    for seq, x in sorted(XML_FILES.items()):
        root = ET.parse(x).getroot()
        ign = []
        for b in root.findall("./ignored_region/box"):
            try:
                l, t, w_, h_ = (float(b.get(k)) for k in ("left", "top", "width", "height"))
                ign.append((l, t, l + w_, t + h_))
            except (TypeError, ValueError):
                schema_errors.append(f"{seq}: ignored_region box without left/top/width/height")
        frames = {}
        for fr in root.findall("./frame"):
            num = fr.get("num")
            if num is None:
                schema_errors.append(f"{seq}: <frame> without num")
                continue
            objs = []
            for t in fr.findall("./target_list/target"):
                b, a = t.find("box"), t.find("attribute")
                if b is None or a is None or a.get("vehicle_type") is None:
                    schema_errors.append(f"{seq} frame {num}: target without box/attribute/vehicle_type")
                    continue
                try:
                    l, tp, w_, h_ = (float(b.get(k)) for k in ("left", "top", "width", "height"))
                except (TypeError, ValueError):
                    schema_errors.append(f"{seq} frame {num}: box without numeric left/top/width/height")
                    continue
                objs.append((a.get("vehicle_type"), (l, tp, l + w_, tp + h_)))
                vtypes[a.get("vehicle_type")] += 1
            frames[int(num)] = objs
        if not frames:
            schema_errors.append(f"{seq}: no <frame> elements")
        img_dir = IMG_DIRS[seq][0]
        jpgs = {p.name for p in img_dir.glob("*.jpg")}
        pattern_ok = all(f"img{n:05d}.jpg" in jpgs for n in list(frames)[:3])
        if not pattern_ok:
            schema_errors.append(f"{seq}: frames not found as img#####.jpg (e.g. {sorted(jpgs)[:3]})")
        missing = [n for n in frames if f"img{n:05d}.jpg" not in jpgs]
        rows.append({"sequence": seq, "split": SEQ_SPLIT_C[seq], "images_on_disk": len(jpgs),
                     "annotated_frames": len(frames), "frames_missing_image": len(missing),
                     "images_without_frame_entry": len(jpgs - {f"img{n:05d}.jpg" for n in frames}),
                     "targets": sum(len(v) for v in frames.values()), "ignored_regions": len(ign)})
        OFFICIAL_ANN[seq] = {"dir": img_dir, "frames": frames, "ignored": ign}
    S = pd.DataFrame(rows)
    print(S.groupby("split")[[c for c in S.columns if c not in ("sequence", "split")]].sum().to_string())
    print("vehicle_type counts:", dict(vtypes))
    unknown = sorted(set(vtypes) - set(CO["vehicle_type_policy"]))
    rep.info.update(per_sequence=rows, vehicle_types=dict(vtypes))
    rep.check("XML matches the expected schema", not schema_errors, schema_errors[:5])
    rep.check("every annotated frame has its image", S["frames_missing_image"].sum() == 0,
              int(S["frames_missing_image"].sum()))
    rep.check("images without a frame entry (excluded, reported)", S["images_without_frame_entry"].sum() == 0,
              int(S["images_without_frame_entry"].sum()), fatal=False)
    rep.check("every vehicle_type has an explicit policy", not unknown, unknown)
    if schema_errors or unknown:
        STATE["ua_detrac_official"].update(status="USER_ACTION_REQUIRED", message="schema/policy mismatch")
        raise UserActionRequired("The official XML does not match the expected schema or has vehicle types "
                                 f"without a policy: {schema_errors[:3]} {unknown}. Nothing was converted. "
                                 "Share one XML file so the parser can be adapted, or extend "
                                 "CFG['ua_detrac_official']['vehicle_type_policy'].")
    rep.finish()

### C3 · Convert, validate and visualise
Splits are by sequence, taken from Part A, and frames are subsampled after the split. Ignored regions are masked exactly as in Part A. The vehicle-type policy maps `van` to `car` and `others` to `truck`. **Check the sample grid for `others`** before trusting that mapping.

In [ ]:
# C3. UA-DETRAC official - convert + YAML + validation + visualisation
if active("ua_detrac_official"):
    assert "OFFICIAL_ANN" in globals(), "Run cell C2 first."
    rep = Report("ua_detrac_official_03_conversion")
    classes = CO["classes"]
    policy = parse_policy(CO["vehicle_type_policy"], classes)
    reset_yolo_root(YOLO_C)
    st, per_type, written = Counter(), defaultdict(Counter), Counter()
    for seq, ann in sorted(OFFICIAL_ANN.items()):
        split = SEQ_SPLIT_C[seq]
        for num in sorted(ann["frames"])[::CO["frame_stride"]]:
            src = ann["dir"] / f"img{num:05d}.jpg"
            with Image.open(src) as im:
                w, h = im.size
            lines, keep = [], []
            for vtype, (x1, y1, x2, y2) in ann["frames"][num]:
                st["targets_seen"] += 1
                kind, cls_id = policy[vtype]
                res, info = clip_box(x1, y1, x2, y2, w, h)
                if res is None:
                    per_type[vtype][f"dropped_invalid:{info}"] += 1
                    continue
                if kind != "map":
                    raise AssertionError(f"policy '{kind}' not supported for vehicle types")
                lines.append(yolo_line(cls_id, res, w, h))
                keep.append(res)
                per_type[vtype]["written_as_" + classes[cls_id]] += 1
            mask = [b for b in (clip_box(*r, w, h)[0] for r in ann["ignored"]) if b is not None]
            stem = f"{seq}__img{num:05d}"
            dst = YOLO_C / "images" / split / f"{stem}.jpg"
            if mask:
                st["images_masked"] += 1
                write_masked_image(src, dst, mask, keep, CO["mask_value"])
            else:
                link_or_copy(src, dst)
            (YOLO_C / "labels" / split / f"{stem}.txt").write_text("\n".join(lines) + ("\n" if lines else ""))
            written[split] += 1
    accounted = sum(v for c in per_type.values() for v in c.values())
    print(pd.DataFrame(per_type).T.fillna(0).astype(int).to_string(), "\n", dict(st), dict(written))
    rep.info.update(per_type={k: dict(v) for k, v in per_type.items()}, written=dict(written))
    rep.check("every target accounted for", accounted == st["targets_seen"], f"{accounted}/{st['targets_seen']}")
    rep.finish()

if CFG["enable"]["ua_detrac_official"] and STATE.get("ua_detrac_official", {}).get("status") in ("PENDING", "PASS"):
    names = CO["classes"]
    yaml_c = write_dataset_yaml(YOLO_C, names, VARIANT_C)
    vrep = validate_yolo_dataset(YOLO_C, names, VARIANT_C)
    mark_prepared(YOLO_C, FP_C, vrep.info["per_split"])
    persist_prepared(VARIANT_C, YOLO_C, FP_C)
    STATE["ua_detrac_official"].update(status="PASS", yaml=str(yaml_c), variant=VARIANT_C,
                                       per_split=vrep.info["per_split"], manifest=vrep.info["manifest"])
    for split in ("train", "val", "test"):
        show_samples(YOLO_C, names, VARIANT_C, split=split, n=8, seed=CFG["seed"])

---
# Data gate
This summarises every dataset and writes `reports/data_gate.json`. The gate passes only if **every enabled dataset** reached `PASS`. The training notebook must load this file and refuse to train when:
* the gate failed;
* a dataset's `fingerprint` differs; or
* the `file_list_sha256` of a split no longer matches the files on disk. This catches files added to or moved between splits after validation.

The gate also records that only `images/train` may feed training and augmentation.

In [ ]:
# D. Data gate
assert "STATE" in globals(), "Run the cells above first."
rows, ready = [], []
for ds in list(REGISTRY) + list(EXCLUDED):
    s = STATE.get(ds, {"status": "NOT RUN"})
    enabled = CFG["enable"].get(ds, False)
    if ds in EXCLUDED:
        status = "EXCLUDED"
    elif not enabled:
        status = "SKIPPED"
    else:
        status = s.get("status", "NOT RUN")
        if s.get("restored") and status != "PASS":
            status = "RESTORED - rerun its validation cell"
    rows.append((ds, status, s.get("per_split", {})))
    if status == "PASS":
        ready.append(ds)
failed = [ds for ds, st, _ in rows if CFG["enable"].get(ds) and st != "PASS"]
gate = {
    "utc": datetime.now(timezone.utc).isoformat(),
    "passed": bool(ready) and not failed,
    "datasets": {ds: {k: STATE[ds].get(k) for k in ("status", "variant", "yaml", "fingerprint", "labels", "per_split", "manifest")}
                 for ds in ready},
    "not_ready": failed,
    "excluded": EXCLUDED,
    "rules": ["Train only on images/train of each YAML.",
              "Training-time augmentation (incl. context augmentation) may only read images/train.",
              "Verify fingerprint and file_list_sha256 before training."],
    "converter_version": CONVERTER_VERSION,
    "ultralytics_version": CFG["ultralytics_version"],
}
(DIRS["reports"] / "data_gate.json").write_text(json.dumps(gate, indent=2, default=str))
for ds, st, per in rows:
    counts = {k: v.get("images") for k, v in per.items()} if per else ""
    print(f"{ds:<20} {st:<38} {counts}")
print(f"\nGate written to {DIRS['reports'] / 'data_gate.json'}")
assert gate["passed"], (f"DATA GATE FAILED - not ready: {failed}. Fix the reported problems; "
                        "do not start training.")
print("DATA GATE PASSED - ready datasets:", ready)